# MART_P07_RESPIRATORIAS_MENSAL

**P07. Quais meses do ano apresentam os maiores picos de internação por doenças
respiratórias no estado?** *(dashboard)*

Doenças respiratórias = diagnóstico principal no capítulo X da CID-10 (`J00`–`J99`,
`FLG_RESPIRATORIA = 'S'`). `NR_RANKING_ANO` = posição do mês dentro do ano (1 = pico).

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO MART_P07_RESPIRATORIAS_MENSAL
    (ANO_MES, NR_ANO, NR_MES, NOME_MES, QTD_INTERNACOES, VL_TOTAL, NR_RANKING_ANO)
SELECT t.ANO_MES, t.NR_ANO, t.NR_MES, t.NOME_MES,
       SUM(f.QTD_INTERNACAO),
       SUM(f.VL_TOTAL),
       RANK() OVER (PARTITION BY t.NR_ANO ORDER BY SUM(f.QTD_INTERNACAO) DESC)
FROM FATO_INTERNACAO f
JOIN DIM_CID c   ON c.SK_CID = f.SK_CID
JOIN DIM_TEMPO t ON t.SK_TEMPO = f.SK_TEMPO_INTERNACAO
WHERE c.FLG_RESPIRATORIA = 'S'
  AND t.NR_ANO IN (SELECT DISTINCT TO_NUMBER(SUBSTR(ANO_MES_CMPT, 1, 4)) FROM FATO_INTERNACAO)
GROUP BY t.ANO_MES, t.NR_ANO, t.NR_MES, t.NOME_MES
"""

In [ ]:
recarregar("MART_P07_RESPIRATORIAS_MENSAL", SQL, origem="FATO_INTERNACAO")

In [ ]:
consultar("SELECT * FROM MART_P07_RESPIRATORIAS_MENSAL FETCH FIRST 10 ROWS ONLY")